# 07 — Desafio 2: Structured Streaming (Bronze → Silver)
## Módulo Desafio 2 | Seções do Exame: S2 + S3

**Contexto do desafio:**  
Substitua o processamento batch do notebook `03` por um pipeline de **Structured Streaming**  
usando `spark.readStream` na Bronze e `writeStream` com `trigger(availableNow=True)` na Silver.

**O que você vai aprender:**
- `readStream` em uma Delta Table como fonte de stream
- `foreachBatch` — permite usar MERGE INTO dentro de um pipeline de streaming
- `trigger(availableNow=True)` — processa todos os dados novos e termina automaticamente
- Checkpoint no Volume do UC — garante processamento incremental entre execuções

**Critério de aceite:**
1. Executar o notebook processa todos os dados novos da Bronze
2. O notebook **termina sozinho** (não fica rodando eternamente)
3. Re-executar o notebook **não reprocessa** dados já processados

---

### ✅ Checklist deste desafio
- [ ] Célula 1 — Configurações e checkpoint path definidos
- [ ] Célula 2 — `readStream` configurado na Bronze
- [ ] Célula 3 — Função `process_batch` com MERGE INTO implementada
- [ ] Célula 4 — `writeStream` com `trigger(availableNow=True)` configurado
- [ ] Célula 4 — Pipeline terminou sozinho (não ficou rodando)
- [ ] Célula 5 — Count da Silver Silver idêntico ao batch (`03`)
- [ ] Re-executei: count da Silver não aumentou (sem reprocessamento)
- [ ] Ingeri novos dados na Bronze e re-executei: count aumentou corretamente

---
## Célula 1 — Configurações

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, ArrayType,
    IntegerType, LongType, FloatType, DoubleType, StringType
)
from delta.tables import DeltaTable

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
dbutils.widgets.text("catalog", "workshop_weather", "Catálogo UC")
CATALOG      = dbutils.widgets.get("catalog")

BRONZE_TABLE     = f"{CATALOG}.bronze.raw_current_weather"
SILVER_TABLE     = f"{CATALOG}.silver.current_weather"
CHECKPOINT_PATH  = f"/Volumes/{CATALOG}/bronze/landing/checkpoints/silver_current"

print(f"Bronze source    : {BRONZE_TABLE}")
print(f"Silver target    : {SILVER_TABLE}")
print(f"Checkpoint path  : {CHECKPOINT_PATH}")
print()
print("ℹ️  O checkpoint persiste entre execuções.")
print("   Para resetar (reprocessar tudo do zero):")
print(f"   dbutils.fs.rm('{CHECKPOINT_PATH}', recurse=True)")

---
## Célula 2 — Schema e `readStream` na Bronze

**Como o Structured Streaming funciona com Delta:**  
`readStream` monitora o transaction log da tabela Bronze.  
Ele sabe exatamente quais versões já foram processadas (via checkpoint)  
e lê apenas as novas — de forma eficiente e sem varredura completa.

In [0]:
# Schema do JSON (mesmo do notebook 03)
WEATHER_SCHEMA = StructType([
    StructField("id",   IntegerType(), True),
    StructField("name", StringType(),  True),
    StructField("coord", StructType([
        StructField("lat", DoubleType(), True),
        StructField("lon", DoubleType(), True),
    ]), True),
    StructField("main", StructType([
        StructField("temp",       FloatType(),   True),
        StructField("feels_like", FloatType(),   True),
        StructField("humidity",   IntegerType(), True),
        StructField("pressure",   IntegerType(), True),
    ]), True),
    StructField("weather", ArrayType(StructType([
        StructField("main",        StringType(), True),
        StructField("description", StringType(), True),
    ])), True),
    StructField("wind", StructType([
        StructField("speed", FloatType(), True),
    ]), True),
    StructField("sys", StructType([
        StructField("country", StringType(), True),
    ]), True),
    StructField("dt", LongType(), True),
])

# readStream: lê novas linhas da Bronze de forma incremental
df_stream = (
    spark.readStream
    .format("delta")
    .table(BRONZE_TABLE)
)

print("✅ readStream configurado.")
print(f"   isStreaming: {df_stream.isStreaming}")  # deve ser True

---
## Célula 3 — Função `process_batch` com MERGE INTO

**Por que `foreachBatch`?**  
`MERGE INTO` não é nativo do Structured Streaming — ele opera em DataFrames batch.  
`foreachBatch` recebe cada micro-batch como um DataFrame batch estático,  
permitindo executar qualquer operação batch (MERGE, criação de views, etc.) dentro do stream.

In [0]:
def process_batch(df_batch, batch_id: int):
    """
    Processa cada micro-batch do stream:
    1. Filtra apenas registros com sucesso
    2. Parseia o JSON com from_json
    3. Extrai e tipifica os campos
    4. Aplica DQ
    5. Executa MERGE INTO na Silver
    """
    # Ignora batches vazios
    if df_batch.isEmpty():
        print(f"Batch {batch_id}: vazio — nenhuma ação necessária.")
        return

    print(f"Processando batch {batch_id}: {df_batch.count()} registros")

    # 1. Filtra sucesso
    df_success = df_batch.filter(
        (F.col("_status") == "success") & F.col("raw_payload").isNotNull()
    )

    # 2. Parseia JSON
    df_parsed = df_success.withColumn(
        "data", F.from_json(F.col("raw_payload"), WEATHER_SCHEMA)
    )

    # 3. Extração de campos
    df_staging = df_parsed.select(
        F.col("data.id")                        .alias("city_id"),
        F.col("data.name")                      .alias("city_name"),
        F.col("data.sys.country")               .alias("country"),
        F.col("data.coord.lat")                 .alias("lat"),
        F.col("data.coord.lon")                 .alias("lon"),
        F.from_unixtime(F.col("data.dt")).cast("timestamp").alias("weather_timestamp"),
        F.col("data.main.temp")                 .alias("temp"),
        F.col("data.main.feels_like")           .alias("feels_like"),
        F.col("data.main.humidity")             .alias("humidity"),
        F.col("data.main.pressure")             .alias("pressure"),
        F.col("data.weather")[0]["main"]        .alias("weather_main"),
        F.col("data.weather")[0]["description"] .alias("weather_description"),
        F.col("data.wind.speed")                .alias("wind_speed"),
        F.col("_batch_id"),
        F.col("_ingestion_timestamp"),
        F.current_timestamp()                   .alias("_processed_timestamp"),
    )

    # 4. DQ
    df_with_dq = df_staging.withColumn(
        "_dq_status",
        F.when(
            F.col("city_id").isNull() | F.col("weather_timestamp").isNull() |
            F.col("temp").isNull() | (F.col("temp") < -80) | (F.col("temp") > 60),
            F.lit("invalid")
        ).otherwise(F.lit("valid"))
    )

    # 5. MERGE INTO via DeltaTable API (não SQL string — mais seguro em foreachBatch)
    delta_target = DeltaTable.forName(spark, SILVER_TABLE)

    (
        delta_target.alias("target")
        .merge(
            df_with_dq.alias("source"),
            "target.city_id = source.city_id AND target.weather_timestamp = source.weather_timestamp"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"  ✅ Batch {batch_id} processado e mergeado na Silver.")

print("✅ Função process_batch definida.")

---
## Célula 4 — `writeStream` com `trigger(availableNow=True)`

**Por que `availableNow=True` e não streaming contínuo?**
- Streaming contínuo ficaria rodando indefinidamente — não é prático em notebooks
- `availableNow=True` processa **todos os dados novos disponíveis** e para automaticamente
- É o padrão recomendado para substituir jobs batch com streaming no Databricks
- Compatible com o Free Edition (não esgota a cota de compute)

In [0]:
# Garante que a tabela Silver existe antes de iniciar o stream
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
    city_id INT, city_name STRING, country STRING,
    lat DOUBLE, lon DOUBLE,
    weather_timestamp TIMESTAMP, temp FLOAT, feels_like FLOAT,
    humidity INT, pressure INT,
    weather_main STRING, weather_description STRING, wind_speed FLOAT,
    _batch_id STRING, _ingestion_timestamp TIMESTAMP,
    _processed_timestamp TIMESTAMP, _dq_status STRING
) USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

# Inicia o stream de processamento
query = (
    df_stream
    .writeStream
    .foreachBatch(process_batch)          # processa cada micro-batch com nossa função
    .option("checkpointLocation", CHECKPOINT_PATH)  # estado incremental persistido
    .trigger(availableNow=True)           # processa tudo disponível e termina
    .start()
)

# Aguarda a conclusão (availableNow garante que vai terminar)
query.awaitTermination()

print()
print("✅ Pipeline de streaming concluído.")
print(f"   Checkpoint salvo em: {CHECKPOINT_PATH}")
print()
print("Re-execute esta célula agora:")
print("  → Se não houve nova ingestão na Bronze, nenhum batch será processado.")
print("  → Isso confirma que o checkpoint está funcionando corretamente.")

---
## Célula 5 — Validação do Processamento Incremental

In [0]:
%sql
-- Verifica o resultado na Silver
SELECT
    city_name,
    ROUND(temp, 1)       AS temp_C,
    weather_main,
    weather_timestamp,
    _processed_timestamp,
    _dq_status
FROM workshop_weather.silver.current_weather
ORDER BY city_name;

In [0]:
# Verifica o conteúdo do checkpoint
print("Conteúdo do diretório de checkpoint:")
try:
    files = dbutils.fs.ls(CHECKPOINT_PATH)
    for f in files:
        print(f"  {f.name}")
    print()
    print("O checkpoint rastreia o offset (versão do Delta log) já processado.")
    print("Na próxima execução, o stream começa exatamente desse ponto.")
except Exception as e:
    print(f"Checkpoint ainda não criado ou caminho não encontrado: {e}")

---
## Célula 6 — Testando o Comportamento Incremental

**Teste completo do critério de aceite:**
1. Execute o notebook Bronze (`01`) para adicionar novos dados
2. Re-execute este notebook
3. Apenas os novos dados devem ser processados

In [0]:
# Conta os registros na Bronze e na Silver para comparar

bronze_count = spark.table(BRONZE_TABLE).filter("_status = 'success'").count()
silver_count = spark.table(SILVER_TABLE).count()

print("=" * 50)
print("  RESUMO DO PIPELINE STREAMING")
print("=" * 50)
print(f"  Bronze (sucesso): {bronze_count} registros")
print(f"  Silver          : {silver_count} registros únicos")
print()
print("  Silver tem menos que Bronze porque MERGE elimina duplicatas")
print("  por (city_id, weather_timestamp) — mesma medição não é inserida 2x.")
print("=" * 50)

---
## ✅ Checklist Final — Desafio 2 concluído?

- [ ] `isStreaming` retornou `True` na célula 2
- [ ] O pipeline terminou sozinho sem intervenção manual
- [ ] Re-executar sem nova ingestão → `process_batch` reportou batches vazios
- [ ] Ingestei novos dados na Bronze → re-executei → Silver atualizou
- [ ] Checkpoint existe em `/Volumes/.../checkpoints/silver_current`
- [ ] Entendi: `foreachBatch` é o padrão para usar MERGE INTO dentro de streaming
- [ ] Entendi: `trigger(availableNow=True)` = streaming que termina sozinho (≠ contínuo)

---
## 💡 Próximos passos — Desafios restantes

**Desafio 1 (Intermediário):**  
Adapte o notebook `03` para usar `table_changes()` com uma tabela `watermark_control`  
em vez de ler toda a Bronze a cada execução.

**Desafio 3 (Expert):**  
Crie `gold.temperature_moving_avg` com window functions de 24h e 72h,  
e configure uma **Genie Space** para consultas em linguagem natural.

---
*Workshop: Engenharia de Dados com Databricks & Delta Lake — OpenWeatherMap API*